# Stage 1 — 10x glomerular segmentation

Resolve one session, build per-odor correlation maps, curate a shared mask, and extract traces.


In [ ]:
%load_ext autoreload
%autoreload 2

import os, shutil, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, "..")

from analysis.session.devshim import LocalGroup
from analysis.session.resolve import resolve_group
from analysis.session.zscore import make_group_keys
from analysis.session.corrcache import build_group_correlation_maps
from analysis.seg_10x.watershed import GLOM_10X_DEFAULTS, scale_params

### SETTINGS TO CHANGE PER SESSION ###
######################################
GROUP_ID = 193
PORTABLE_MASK_BUNDLE = None  # optional server HDF5 for extraction-only runs if not in default location
MANIPULATION = ""
APPROVED_ONLY = True
EXCLUDE_ACQ = []
SEPARATE_BY_CONDITION = False
DETREND = True
PRE_S, POST_S, SIGMA_PX = 2.0, 3.0, 0.5
#######################################

# Keep large temporary writes local; set these roots once per machine.
SCRATCH = Path(os.environ.get("ODYN_SCRATCH_ROOT", Path.home() / "odyn_scratch"))
MAIN = Path(os.environ.get("ODYN_IMAGING_ROOT", "/Volumes/MossLab/ImagingData"))
SCRATCH.mkdir(parents=True, exist_ok=True)
CORR_CACHE = SCRATCH / "correlation_cache" / f"group{GROUP_ID}"

# A local metadata copy avoids repeated slow reads from the server. The resolver
# uses frame sync when available and otherwise the best acquisition timing source.
group = LocalGroup(
    MAIN / ".odyn" / "odyn.db", 
    MAIN,
    snapshot_to=SCRATCH / "odyn_snapshot.db", max_age_s=1800,
    refresh=True,
)
session = resolve_group(
    group, group_id=GROUP_ID, manipulation=MANIPULATION,
    approved_only=APPROVED_ONLY, exclude_acq_ids=tuple(EXCLUDE_ACQ),
)
print(session.summary())




In [ ]:
import bokeh.plotting as bpl
from analysis.session.bokeh import stop_notebook_servers
def vscode_url(port):
    if port is None:
        return "*"
    return f"http://localhost:{port}/"

if not hasattr(bpl, "_odyn_original_show"):
    bpl._odyn_original_show = bpl.show

def vscode_show(obj, *args, **kwargs):
    if callable(obj):
        stop_notebook_servers()
        kwargs.setdefault("notebook_url", vscode_url)
        kwargs.setdefault("port", 5008)

    return bpl._odyn_original_show(obj, *args, **kwargs)

bpl.show = vscode_show

print("Bokeh configured for VS Code port 5008 with automatic server cleanup")


## Correlation maps

Each trial is baseline-z-scored, averaged within odor group, and reduced to an 8-neighbour local-correlation map. On a cache miss, progress bars report completed/total acquisitions during streaming and completed/total odor groups during map calculation, including rate and ETA. Cache hits load immediately without displaying a bar. Accumulators and correlation caches stay on local scratch to avoid server traffic.


In [ ]:
keys = make_group_keys(
    session.odor_ids, session.states,
    separate_by_condition=SEPARATE_BY_CONDITION,
)
WORK = SCRATCH / f"nb_work_{GROUP_ID}"

corr_by_odor, corr_meta = build_group_correlation_maps(
    session.paths,
    odor_on_frames=session.odor_on_frames,
    odor_off_frames=session.odor_off_frames,
    group_keys=keys,
    frame_rate=session.frame_rate,
    pre_s=PRE_S,
    post_s=POST_S,
    spatial_sigma_px=SIGMA_PX,
    work_dir=WORK,
    cache_dir=CORR_CACHE,
    progress=True,
)
print(f"{len(corr_by_odor)} maps; cache {corr_meta['cache']}")


## Batch: warm correlation maps for several groups

Optional. List the group ids to precompute, and this streams each session once and writes its correlation cache. Afterwards, setting `GROUP_ID` to any of them and re-running the cells above is a cache hit, so curation starts immediately. The cache key includes the movie paths, `PRE_S`, `POST_S`, `SIGMA_PX`, and `SEPARATE_BY_CONDITION`, so leave the settings block alone between the batch run and curation or every session re-streams.

In [ ]:
GROUP_IDS = [191, 193]
EXCLUDE_BY_GROUP = {}  # per-session exclusions, e.g. {193: [3]}

def warm_correlation_maps(group_id, *, keep=False):
    batch_session = resolve_group(
        group, group_id=group_id, manipulation=MANIPULATION,
        approved_only=APPROVED_ONLY,
        exclude_acq_ids=tuple(EXCLUDE_BY_GROUP.get(group_id, ())),
    )
    batch_keys = make_group_keys(
        batch_session.odor_ids, batch_session.states,
        separate_by_condition=SEPARATE_BY_CONDITION,
    )
    maps, meta = build_group_correlation_maps(
        batch_session.paths,
        odor_on_frames=batch_session.odor_on_frames,
        odor_off_frames=batch_session.odor_off_frames,
        group_keys=batch_keys,
        frame_rate=batch_session.frame_rate,
        pre_s=PRE_S,
        post_s=POST_S,
        spatial_sigma_px=SIGMA_PX,
        work_dir=SCRATCH / f"nb_work_{group_id}",
        cache_dir=SCRATCH / "correlation_cache" / f"group{group_id}",
        progress=True,
    )
    if keep:
        return batch_session, maps, meta
    return f"{len(maps)} maps ({meta['cache']})"

batch_status = {}
for gid in GROUP_IDS:
    try:
        batch_status[gid] = warm_correlation_maps(gid)
    except Exception as error:
        batch_status[gid] = f"FAILED {type(error).__name__}: {error}"
    print(gid, batch_status[gid])

## Segmentation GUI

The GUI moves through three states:

1. **Tune** — adjust segmentation for all odors or override one odor. Threshold controls sensitivity; adaptive thresholding handles uneven backgrounds; diameter limits set ROI size; peak distance controls watershed splitting; border excludes edge artifacts.
2. **Merge** — freeze segmentation and combine matching detections. Minimum overlap controls matching, minimum detections requires support across odors, and consensus fraction controls how much of the overlapping footprint is retained.
3. **Curate** — freeze parameters, then add, delete, or exclude ROIs on the merged mask. Returning to an earlier state discards later edits.

Manual additions grow from watershed seeds. **Save local checkpoint (not publish)** stores the curated mask and settings under `ODYN_SCRATCH_ROOT`. You must then run **Final mask** to publish the portable bundle used by batch extraction.


In [ ]:
from analysis.seg_10x.gui import launch

params = scale_params(GLOM_10X_DEFAULTS, to_um_per_px=session.um_per_px)
gui = launch(
    corr_by_odor,
    save_path=SCRATCH / f"masks_group{GROUP_ID}.h5",
    params=params,
)
gui.group_id = GROUP_ID  # prevent reuse after changing sessions


## Final mask

Run after GUI curation. This automatically publishes a compact mask-only HDF5 to the shared session output directory. It contains the curated labels, per-odor masks, reference image, segmentation/merge parameters, and curation record needed for later trace extraction. With `PORTABLE_MASK_BUNDLE=None`, another kernel or computer automatically discovers the newest published bundle; set it to a path only to override that choice.


In [ ]:
from analysis.session.finalize import mask_hash
from analysis.session.masks import (
    background_image, load_latest_mask, load_mask_bundle,
    load_10x_working_mask,
    save_mask_bundle, save_mask_overlay,
)
from analysis.session.store import session_filename

def output_path(kind, suffix):
    return session.output_dir / session_filename(
        group_id=session.group_id, exp_name=session.exp_name,
        kind=kind, suffix=suffix,
    )

if PORTABLE_MASK_BUNDLE is not None:
    saved = load_mask_bundle(PORTABLE_MASK_BUNDLE)
else:
    published_bundles = sorted(session.output_dir.glob(
        f'group{GROUP_ID}_*_10x_masks_processed_*.h5'
    ))
    working_checkpoint = SCRATCH / f'masks_group{GROUP_ID}.h5'
    saved = (load_mask_bundle(published_bundles[-1])
             if published_bundles else
             load_10x_working_mask(working_checkpoint)
             if working_checkpoint.exists() else
             load_latest_mask(session.output_dir))

if ("gui" in dir() and getattr(gui, 'group_id', None) == GROUP_ID
        and gui.state.phase == "curate"):
    labels = gui.state.curated_mask()
    masks_by_group = gui.state.segment_all()
    reference = background_image(corr_by_odor)
    params = dict(gui.state.shared)
    merge_params = dict(gui.state.merge_params)
    curation_record = gui.state.summary()
    source = "curated"
elif saved is not None:
    labels = saved["labels"]
    masks_by_group = saved.get("per_group", {})
    reference = saved.get("reference")
    if reference is None and 'corr_by_odor' in dir():
        reference = background_image(corr_by_odor)
    config = saved.get("config", {})
    params = config.get("segmentation", {})
    merge_params = config.get("merge", {})
    curation_record = config.get("curation")
    source = (f"portable: {saved['path'].name}"
              if saved.get('source') != '10x GUI working checkpoint'
              else f"working checkpoint: {saved['path'].name}")
elif ("gui" in dir() and getattr(gui, 'group_id', None) == GROUP_ID):
    labels = gui.state.merged().labels
    masks_by_group = gui.state.segment_all()
    reference = background_image(corr_by_odor)
    params = dict(gui.state.shared)
    merge_params = dict(gui.state.merge_params)
    curation_record = None
    source = "automatic"
else:
    raise FileNotFoundError("Set PORTABLE_MASK_BUNDLE or run the segmentation GUI.")

if reference is None:
    raise ValueError("The portable mask bundle has no reference image.")

published_new_bundle = not source.startswith("portable:")
if published_new_bundle:
    bundle = save_mask_bundle(
        output_path("10x_masks", ".h5"), labels,
        per_group_masks=masks_by_group,
        reference=reference,
        config={"segmentation": params, "merge": merge_params,
                "curation": curation_record},
    )
    source += f"; published {bundle.name}"
    print(f"published portable 10x mask bundle: {bundle}")
    png = save_mask_overlay(output_path("masks", ".png"), reference, labels)
    print(png.name)
else:
    print('loaded existing portable mask bundle; segmentation overlay not regenerated')

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(reference, cmap="gray", vmin=np.percentile(reference, 1),
          vmax=np.percentile(reference, 99.5))
ax.contour(labels > 0, levels=[0.5], colors="magenta", linewidths=0.6)
ax.set(title=f"{int(labels.max())} ROIs — {source}", xticks=[], yticks=[])



## Load the batch-extracted component round

Trace extraction is now a separate batch stage (`python -m analysis.batch_10x --execute`). This cell reloads the extracted component round whose mask matches the published bundle. Component traces are used only to review joins; final analysis and QC below contain joined units and remaining singletons.

MATLAB reverses HDF5 dimensions; restore the Python axis order when reading:

```matlab
file = "group..._processed_YYYYMMDD.h5";
labels = h5read(file, "/masks/labels").';
roi = permute(h5read(file, "/traces/roi"), [3 2 1]); % ROI × trial × frame
time_s = h5read(file, "/traces/time_s");
```


In [ ]:
from analysis.session.finalize import mask_hash
from analysis.session.h5io import open_h5
from analysis.session.store import find_rounds

current_hash = mask_hash(labels)
rounds = []
for candidate in find_rounds(session.output_dir):
    with open_h5(candidate) as handle:
        if ('traces/roi' in handle
                and handle.attrs.get('mask_hash') == current_hash):
            rounds.append(candidate)
if not rounds:
    raise FileNotFoundError(
        'No extracted round matches this published mask. Run analysis.batch_10x first.'
    )
round_path = rounds[-1]
GROUPS_PATH = session.output_dir / f'group{GROUP_ID}_10x_roi_groups.json'
print('component round:', round_path)
print('reviewed joins:', GROUPS_PATH)


## Review neighboring, correlated ROI joins

Nearby pairs are ranked using canonical odor/post-odor trace correlation. The suggestions are diagnostic only: click fragments to select them, assign a join ID, and save. Unassigned ROIs remain singleton final units.

In [ ]:
from analysis.seg_10x.grouping import prepare_joining, launch_joining

JOINING = {'max_gap_px': 8.0, 'min_correlation': 0.70, 'max_lag_frames': 1}
joining_state = prepare_joining(
    round_path, reference, params=JOINING, groups_path=GROUPS_PATH,
)
display(joining_state.candidates.head(50))
joining_gui = launch_joining(joining_state, GROUPS_PATH)


## Final joined traces and QC

This recomputes traces by pixel-weighting raw component fluorescence within each reviewed join before detrending and normalization. The final HDF5 and every QC panel contain only joined glomerular units and remaining singletons.

In [ ]:
import json
from analysis.seg_10x.grouped_qc import finalize_grouped_10x

BASELINE_SD_MODE = 'pre_block_pooled'
qc = finalize_grouped_10x(
    round_path, GROUPS_PATH, reference=reference,
    baseline_sd_mode=BASELINE_SD_MODE,
)
print(json.dumps(qc, indent=2))


## Remove caches after final QC

Cleanup refuses to run unless the grouped HDF5 and all final QC artifacts exist and are nonempty.

In [ ]:
from analysis.seg_10x.grouped_qc import cleanup_10x_caches

removed = cleanup_10x_caches(
    session.output_dir, SCRATCH, GROUP_ID, qc_outputs=qc,
)
print('removed caches:', *removed, sep='\n  ')
